In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
data = [
"artificial intelligence systems learn patterns from data",
"sequence models process information step by step",
"recurrent neural networks are useful for sequence prediction",
"lstm networks handle long term dependencies",
"deep learning models improve sequence learning",
"generative models create new samples from learned patterns",
"language models predict the next word in a sentence",
"sequence generation is used in chatbots and assistants",
"machine learning helps computers learn automatically",
"training data improves model accuracy",
"neural networks simulate human brain structures",
"optimization algorithms improve learning efficiency",
"technology is transforming modern education",
"online learning platforms use artificial intelligence",
"students benefit from intelligent tutoring systems",
"automation improves productivity and decision making"
]

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)

total_words = len(tokenizer.word_index) + 1

In [4]:
input_sequences = []

for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_seq_len = max(len(x) for x in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)


In [5]:
def generate_text(model, seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)[0]
        output_word = tokenizer.index_word.get(predicted, "")

        seed_text += " " + output_word

    return seed_text

In [6]:
print("\n===== LSTM MODEL =====\n")

lstm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(total_words, 64, input_length=max_seq_len-1),
    tf.keras.layers.LSTM(100),
    tf.keras.layers.Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

lstm_model.fit(X, y, epochs=200, verbose=1)

print("\nGenerated Text (LSTM):")
print(generate_text(lstm_model, "deep learning models", 5))


===== LSTM MODEL =====



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.0000e+00 - loss: 4.3593
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0795 - loss: 4.3497 
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0795 - loss: 4.3411
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0795 - loss: 4.3315
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0682 - loss: 4.3181
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.0568 - loss: 4.3006
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.2720
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.2250
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.1500
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.0568 - loss: 4.0922
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.0568 - loss: 4.0861 
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.056

In [7]:
print("\n===== TRANSFORMER MODEL =====\n")

# Positional Encoding
def positional_encoding(length, d_model):
    positions = np.arange(length)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(10000, (2 * (dims//2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)


===== TRANSFORMER MODEL =====



In [8]:
def transformer_block(inputs, head_size, num_heads, ff_dim):
    attention = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=head_size
    )(inputs, inputs)

    attention = tf.keras.layers.Dropout(0.1)(attention)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(inputs + attention)

    ff = tf.keras.layers.Dense(ff_dim, activation='relu')(x)
    ff = tf.keras.layers.Dense(inputs.shape[-1])(ff)

    ff = tf.keras.layers.Dropout(0.1)(ff)
    return tf.keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)

In [9]:
inputs = tf.keras.Input(shape=(max_seq_len-1,))
embedding_dim = 64

x = tf.keras.layers.Embedding(total_words, embedding_dim)(inputs)

# Add positional encoding
pos_encoding = positional_encoding(max_seq_len-1, embedding_dim)
x = x + pos_encoding[:, :max_seq_len-1, :]

# Transformer block
x = transformer_block(x, head_size=64, num_heads=2, ff_dim=128)

# Global pooling
x = tf.keras.layers.GlobalAveragePooling1D()(x)

x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(total_words, activation='softmax')(x)

transformer_model = tf.keras.Model(inputs, outputs)

transformer_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

transformer_model.fit(X, y, epochs=200, verbose=1)

print("\nGenerated Text (Transformer):")
print(generate_text(transformer_model, "machine learning", 5))

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.0114 - loss: 4.4166
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0227 - loss: 4.3257    
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0682 - loss: 4.2726
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0568 - loss: 4.2527
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0568 - loss: 4.2448
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0568 - loss: 4.2057
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0568 - loss: 4.2081
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0455 - loss: 4.1899
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0568 - loss: 4.1756
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0568 - loss: 4.1718
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.0341 - loss: 4.1471    
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.02